In [ ]:
audio_file = "../Projects/whisper-transcribe/Worst Case.mp3"

In [ ]:
pip install transformers>=5.4.0 torch huggingface_hub soundfile librosa sentencepiece protobuf accelerate

In [ ]:
from transformers import AutoProcessor, CohereAsrForConditionalGeneration
from transformers.audio_utils import load_audio
from huggingface_hub import hf_hub_download
import time
processor = AutoProcessor.from_pretrained("CohereLabs/cohere-transcribe-03-2026")
model = CohereAsrForConditionalGeneration.from_pretrained("CohereLabs/cohere-transcribe-03-2026", device_map="auto")

audio = load_audio(audio_file, sampling_rate=16000)

inputs = processor(audio, sampling_rate=16000, return_tensors="pt", language="en")
inputs.to(model.device, dtype=model.dtype)

outputs = model.generate(**inputs, max_new_tokens=256)
start_time = time.time()
cohere_text = processor.decode(outputs, skip_special_tokens=True)
end_time = time.time()
cohere_time = end_time - start_time
print(cohere_text)


In [ ]:
pip install qwen_asr torch flash-attn --no-build-isolation

In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-1.7B",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)

start_time = time.time()
results = model.transcribe(
    audio=audio_file,
    language=None, # set "English" to force the language
)
end_time = time.time()

print(results[0].language)
qwen_text = results[0].text
qwen_time = end_time - start_time
print(results[0].text)

In [ ]:
pip install "nemo_toolkit[asr,tts] @ git+https://github.com/NVIDIA/NeMo.git"

In [ ]:
from nemo.collections.speechlm2.models import SALM

model = SALM.from_pretrained('nvidia/canary-qwen-2.5b')

start_time = time.time()
answer_ids = model.generate(
    prompts=[
        [{"role": "user", "content": f"Transcribe the following: {model.audio_locator_tag}", "audio": [audio_file]}]
    ],
    max_new_tokens=128,
)
end_time = time.time()
canary_time = end_time - start_time
print(model.tokenizer.ids_to_text(answer_ids[0].cpu()))

In [ ]:
print(f"Canary time: {canary_time:.2f} seconds")
print(f"Cohere time: {cohere_time:.2f} seconds")